# Workshop — RAG sobre papers académicos con LangChain y NVIDIA NIM

En este workshop construirás un sistema RAG completo alimentado con dos papers fundamentales:

- **Attention Is All You Need** — Vaswani et al. (2017)
- **Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks** — Lewis et al. (2020)

El notebook descarga los PDFs desde arXiv, extrae su texto, crea embeddings y responde preguntas usando únicamente el contenido indexado.

**Stack:** MarkItDown · LangChain · NVIDIA NIM · Qdrant

| Parte | Tarea |
|---|---|
| 0 | Instalar y configurar Colab |
| 1 | Descargar y convertir PDFs con MarkItDown |
| 2 | Crear `Document` y aplicar chunking |
| 3 | Generar embeddings NVIDIA e indexar en Qdrant |
| 4 | Construir el pipeline RAG con LCEL |
| 5 | Evaluar mediante preguntas con *ground truth* |

**Modelos:**
- Embeddings: `nvidia/nemotron-3-embed-1b`
- Generación: `nvidia/nemotron-3.5-super-vl-120b-a12b` (mi cuenta NVIDIA no tiene habilitado `nemotron-3-super-120b-a12b`, responde 404; este es del mismo tamaño, 120B con 12B activos)


In [ ]:
# Dependencias para Google Colab
%pip install -qU "markitdown[pdf]" langchain langchain-core langchain-text-splitters langchain-nvidia-ai-endpoints langchain-qdrant qdrant-client python-dotenv requests


In [ ]:
import os
import getpass
import time
import random
from pathlib import Path
from typing import List

import requests
from dotenv import load_dotenv

# Busca un .env en la carpeta actual y en las dos carpetas padre
# (en este repo el .env está un nivel arriba de notebooks/).
for _d in [Path.cwd(), *list(Path.cwd().parents)[:2]]:
    if (_d / ".env").exists():
        load_dotenv(_d / ".env")
        break

NVIDIA_API_KEY = None
try:
    from google.colab import userdata          # Colab Secrets
    NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")
except Exception:
    NVIDIA_API_KEY = None
if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = os.getenv("NVIDIA_API_KEY")

if not NVIDIA_API_KEY:
    NVIDIA_API_KEY = getpass.getpass("NVIDIA_API_KEY: ").strip()
if not NVIDIA_API_KEY:
    raise ValueError("Falta NVIDIA_API_KEY. Créala en https://build.nvidia.com/")

os.environ["NVIDIA_API_KEY"] = NVIDIA_API_KEY   # la clave nunca se imprime

# CAMBIO DE MODELO: el original del taller es "nvidia/nemotron-3-super-120b-a12b",
# pero mi cuenta NVIDIA no lo tiene habilitado (responde 404).
# Uso "nvidia/nemotron-3.5-super-vl-120b-a12b": mismo tamaño (120B, 12B activos).
# Es un modelo con razonamiento, por eso subo max_completion_tokens más abajo.
GENERATION_MODEL = "nvidia/nemotron-3.5-super-vl-120b-a12b"
EMBEDDING_MODEL = "nvidia/nemotron-3-embed-1b"

print("✅ Configuración NVIDIA NIM lista")
print("   Generación:", GENERATION_MODEL)
print("   Embeddings:", EMBEDDING_MODEL)


In [ ]:
# Ayudas para compartir la clave NVIDIA sin ráfagas: reintentos con backoff ante 429/5xx
# y una pausa corta entre llamadas. Llamadas siempre secuenciales.
TRANSIENT = ("429", "500", "502", "503", "504", "timed out", "timeout", "Timeout",
             "rate limit", "Rate limit", "Too Many", "temporarily", "Connection", "empty response")
PAUSE = 0.7


def retry_call(fn, *args, tries: int = 7, base: float = 2.0, **kwargs):
    for attempt in range(tries):
        try:
            return fn(*args, **kwargs)
        except Exception as exc:
            transient = any(t in str(exc) for t in TRANSIENT)
            if attempt == tries - 1 or not transient:
                raise
            wait = base * (2 ** attempt) + random.random()
            print(f"  ⏳ error transitorio ({type(exc).__name__}); reintento {attempt + 1} en {wait:.0f}s")
            time.sleep(wait)


def _invoke_nonempty(chain, question):
    out = chain.invoke(question)
    if not out or not out.strip():
        raise RuntimeError("empty response (el modelo gastó los tokens razonando)")
    return out


def ask(chain, question: str) -> str:
    """Invoca la cadena RAG con reintentos y pausa."""
    out = retry_call(_invoke_nonempty, chain, question)
    time.sleep(PAUSE)
    return out


def retrieve(retr, question: str):
    out = retry_call(retr.invoke, question)
    time.sleep(0.3)
    return out


def add_in_batches(store, docs, batch_size: int = 50):
    """Indexa por lotes para no saturar la API de embeddings."""
    ids = []
    for i in range(0, len(docs), batch_size):
        ids += retry_call(store.add_documents, docs[i:i + batch_size])
        time.sleep(0.5)
    return ids


---
## Parte 1 — Descargar y convertir los PDFs

La celda descarga los artículos únicamente si todavía no existen en `papers/`. Después, MarkItDown convierte cada PDF a texto conservando buena parte de su estructura.

> Si una sesión de Colab se reinicia, los archivos temporales se pierden y se descargarán nuevamente.


In [ ]:
from markitdown import MarkItDown

# Usa papers/ si ya existe (en este repo está un nivel arriba de notebooks/);
# si no existe en ningún lado, la crea y descarga los PDFs (caso Colab).
DOCS_DIR = next((p for p in (Path("papers"), Path("../papers")) if p.exists()), Path("papers"))
DOCS_DIR.mkdir(parents=True, exist_ok=True)

PAPERS = {
    "attention_paper.pdf": "https://arxiv.org/pdf/1706.03762",
    "rag_paper.pdf": "https://arxiv.org/pdf/2005.11401",
}


def download_file(url: str, destination: Path) -> None:
    if destination.exists() and destination.stat().st_size > 10_000:
        print(f"↪ Ya existe: {destination.name}")
        return

    response = requests.get(url, timeout=90)
    response.raise_for_status()
    if not response.content.startswith(b"%PDF"):
        raise ValueError(f"La descarga de {url} no produjo un PDF válido")
    destination.write_bytes(response.content)
    print(f"⬇ Descargado: {destination.name} ({destination.stat().st_size / 1e6:.1f} MB)")


for filename, url in PAPERS.items():
    download_file(url, DOCS_DIR / filename)

md_converter = MarkItDown(enable_plugins=False)


def load_pdf(path: str | Path) -> str:
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f"Archivo no encontrado: {path}")
    result = md_converter.convert(str(path))
    text = result.text_content
    if not text or not text.strip():
        raise ValueError(f"No se pudo extraer texto de {path.name}")
    return text


attention_text = load_pdf(DOCS_DIR / "attention_paper.pdf")
rag_text = load_pdf(DOCS_DIR / "rag_paper.pdf")

print(f"✅ attention_paper.pdf → {len(attention_text):,} caracteres")
print(f"✅ rag_paper.pdf       → {len(rag_text):,} caracteres")
print(f"\nInicio del paper de Attention:\n{attention_text[:300]}...")


---
## Parte 2 — Convertir a `Document` y aplicar chunking

LangChain utiliza objetos `Document` con dos campos principales:

- `page_content`: contenido textual del fragmento.
- `metadata`: título, archivo de origen e índice del fragmento.

Usaremos fragmentos de 900 caracteres con solapamiento de 120. Es un punto de partida didáctico; no es un valor universal.


In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_core.documents import Document


def pdf_to_documents(text: str, title: str, source: str) -> List[Document]:
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=900,
        chunk_overlap=120,
        separators=["\n\n", "\n", ".", " "],
    )
    documents = []
    for index, chunk in enumerate(splitter.split_text(text)):
        if len(chunk.strip()) < 30:
            continue
        documents.append(Document(
            page_content=chunk,
            metadata={"title": title, "source": source, "chunk_index": index},
        ))
    return documents


attention_docs = pdf_to_documents(
    attention_text,
    title="Attention Is All You Need",
    source="attention_paper.pdf",
)
rag_docs = pdf_to_documents(
    rag_text,
    title="Retrieval-Augmented Generation",
    source="rag_paper.pdf",
)
all_docs = attention_docs + rag_docs

print(f"Attention → {len(attention_docs)} chunks")
print(f"RAG       → {len(rag_docs)} chunks")
print(f"Total     → {len(all_docs)} chunks")
print("\nEjemplo de metadatos:", all_docs[0].metadata)


---
## Parte 3 — Embeddings NVIDIA e índice Qdrant

`NVIDIAEmbeddings` distingue automáticamente entre embeddings de documentos y consultas mediante `embed_documents()` y `embed_query()`. La dimensión se obtiene directamente del modelo para evitar valores codificados manualmente.

Qdrant se ejecuta en memoria; al reiniciar el runtime se reconstruye el índice.


In [ ]:
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
from qdrant_client import QdrantClient
from qdrant_client.models import Distance, VectorParams
from langchain_qdrant import QdrantVectorStore

COLLECTION = "academic_papers_nvidia"

embeddings = NVIDIAEmbeddings(
    model=EMBEDDING_MODEL,
    truncate="END",
)
EMBEDDING_DIM = len(embeddings.embed_query("dimension probe"))

qdrant_client = QdrantClient(":memory:")
qdrant_client.create_collection(
    collection_name=COLLECTION,
    vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE),
)

vectorstore = QdrantVectorStore(
    client=qdrant_client,
    collection_name=COLLECTION,
    embedding=embeddings,
)
ids = add_in_batches(vectorstore, all_docs)

print(f"✅ {len(ids)} chunks indexados en Qdrant")
print(f"✅ Dimensión de embeddings: {EMBEDDING_DIM}")


---
## Parte 4 — Pipeline RAG con LCEL

El pipeline conecta recuperación, construcción del prompt, generación con Nemotron y extracción del texto final. El prompt obliga al modelo a reconocer cuando el contexto recuperado no contiene la respuesta.


In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatNVIDIA(
    model=GENERATION_MODEL,
    temperature=0.2,
    top_p=0.95,
    max_completion_tokens=1500,   # modelo con razonamiento: 800 podía quedarse corto
)

retriever = vectorstore.as_retriever(
    search_type="similarity_score_threshold",
    search_kwargs={"k": 5, "score_threshold": 0.25},
)


def format_docs(documents: List[Document]) -> str:
    return "\n\n---\n\n".join(
        f"[{doc.metadata.get('title', 'Untitled')} | {doc.metadata.get('source', '')}]"
        f"\n{doc.page_content}"
        for doc in documents
    )


prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        "Answer using only the supplied academic-paper excerpts. "
        "If the excerpts do not contain enough evidence, say so explicitly. "
        "Answer in the same language as the question and be concise. "
        "Mention the paper title supporting the answer.",
    ),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])

rag_chain = (
    {
        "context": retriever | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

test_question = "What is the Transformer architecture?"
test_answer = ask(rag_chain, test_question)
print("Pregunta:", test_question)
print("Respuesta:", test_answer)


---
## Parte 5 — Quiz sobre los papers

Ejecuta cada pregunta y compara la respuesta del RAG con el *ground truth*. Las preguntas están en inglés para coincidir con el idioma de los artículos; el prompt pide responder en el mismo idioma de la consulta.

Además de la corrección factual, observa si el contexto recuperado proviene del paper adecuado.


In [ ]:
# ── Pregunta 1 ────────────────────────────────────────────────
q1 = "What BLEU score did the Transformer (big model) achieve on WMT 2014 English-to-German?"

answer1 = ask(rag_chain, q1)
print(f"Q: {q1}")
print(f"A: {answer1}")

<details>
<summary>✅ Ground Truth — Pregunta 1</summary>

El Transformer (big model) obtuvo un BLEU score de **28.4** en la tarea de
traducción WMT 2014 English-to-German, superando los mejores resultados
previos (incluyendo ensembles) por más de 2 puntos BLEU.
*(Fuente: Abstract y Section 6.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 2 ────────────────────────────────────────────────
q2 = "How does RAG combine parametric and non-parametric memory?"

answer2 = ask(rag_chain, q2)
print(f"Q: {q2}")
print(f"A: {answer2}")

<details>
<summary>✅ Ground Truth — Pregunta 2</summary>

RAG combina **memoria paramétrica** (un modelo seq2seq pre-entrenado, BART-large)
con **memoria no-paramétrica** (un índice denso de vectores de Wikipedia).
El retriever (DPR) recupera los top-K documentos relevantes para la query,
y el generador (BART) los usa como contexto adicional para producir la respuesta.
Ambos componentes se entrenan end-to-end de forma conjunta.
*(Fuente: Abstract y Section 2 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 3 ────────────────────────────────────────────────
q3 = "What is the purpose of the scaling factor sqrt(dk) in Scaled Dot-Product Attention?"

answer3 = ask(rag_chain, q3)
print(f"Q: {q3}")
print(f"A: {answer3}")

<details>
<summary>✅ Ground Truth — Pregunta 3</summary>

El factor de escala **1/√dk** se aplica para contrarrestar el efecto de que,
con valores grandes de dk, los productos punto crecen en magnitud y empujan
la función softmax hacia regiones con gradientes muy pequeños.
Esto haría el entrenamiento inestable. La división por √dk mantiene los
productos punto en un rango con gradientes útiles.
*(Fuente: Section 3.2.1 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta 4 ────────────────────────────────────────────────
q4 = "What are the two RAG model variants proposed in the paper and how do they differ?"

answer4 = ask(rag_chain, q4)
print(f"Q: {q4}")
print(f"A: {answer4}")

<details>
<summary>✅ Ground Truth — Pregunta 4</summary>

Los dos variantes son:
- **RAG-Sequence**: usa el **mismo documento recuperado** para generar toda
  la secuencia de salida. Trata el documento como una variable latente única.
- **RAG-Token**: puede usar un **documento diferente para cada token** generado,
  permitiendo combinar información de múltiples fuentes en una sola respuesta.

RAG-Token es más flexible pero más costoso computacionalmente.
*(Fuente: Section 2.1 — RAG paper)*
</details>

In [ ]:
# ── Pregunta 5 ────────────────────────────────────────────────
q5 = "How many attention heads does the base Transformer model use, and what are the key and value dimensions per head?"

answer5 = ask(rag_chain, q5)
print(f"Q: {q5}")
print(f"A: {answer5}")

<details>
<summary>✅ Ground Truth — Pregunta 5</summary>

El Transformer base usa **h = 8** attention heads en paralelo.
Para cada cabeza, las dimensiones son **dk = dv = dmodel/h = 512/8 = 64**.
El costo computacional total es similar al de single-head attention con
dimensionalidad completa, gracias a las dimensiones reducidas por cabeza.
*(Fuente: Section 3.2.2 — Attention Is All You Need)*
</details>

---
## Bonus — Pregunta entre ambos papers

Esta pregunta exige conectar información de los dos artículos. Además de generar la respuesta, inspeccionaremos los títulos y archivos de los fragmentos recuperados.


In [ ]:
q_bonus = ("The RAG paper uses FAISS for fast retrieval with an HNSW approximation. "
           "The Attention paper proposes the Transformer architecture. "
           "What seq2seq model does RAG use as its generator component, "
           "and is it based on the Transformer?")

answer_bonus = ask(rag_chain, q_bonus)

# Mostrar también qué chunks recuperó
docs_retrieved = retrieve(retriever, q_bonus)
sources = [(d.metadata["title"], d.metadata["source"]) for d in docs_retrieved]

print(f"Q: {q_bonus}\n")
print(f"A: {answer_bonus}\n")
print(f"Fuentes recuperadas:")
for title, source in sources:
    print(f"  [{title}] — {source}")

<details>
<summary>✅ Ground Truth — Bonus</summary>

RAG usa **BART-large** como generador (Section 2.3 del RAG paper).
BART es un modelo seq2seq pre-entrenado basado en la arquitectura **Transformer**
(referencia [58] en el RAG paper = el paper "Attention Is All You Need").
El RAG paper cita explícitamente: "BART-large, a pre-trained seq2seq transformer".

Esta pregunta demuestra el valor de RAG: el sistema conecta información de
dos documentos distintos para construir una respuesta completa.
</details>

---
## B1 · Evidencia: contexto recuperado y verificación contra el *ground truth*

Para cada pregunta imprimo los 5 fragmentos que llegaron al modelo (con su score de relevancia que reporta LangChain) y reviso con un criterio **estricto y automático** si la respuesta contiene los datos del *ground truth*. El criterio es el mismo que usaré en B2: la respuesta cuenta como acierto solo si contiene **todos** los datos clave de la lista (no cuenta que "suene bien").

| Pregunta | Datos que debe contener la respuesta |
|---|---|
| Q1 | `28.4` |
| Q2 | memoria no paramétrica + `BART` + Wikipedia / índice denso / DPR |
| Q3 | `softmax` + `gradient(s)` + dot products grandes (large / grow / magnitude) |
| Q4 | `RAG-Sequence` + `RAG-Token` + mismo documento + documento distinto por token |
| Q5 | 8 cabezas + `64` |
| Bonus | `BART` + `Transformer` |

In [ ]:
import re

QUIZ = {"Q1": q1, "Q2": q2, "Q3": q3, "Q4": q4, "Q5": q5}

# Todos los patrones de la lista deben aparecer en la respuesta (sin distinguir mayúsculas).
GT_PATTERNS = {
    "Q1": [r"28\.4"],
    "Q2": [r"non-?\s?parametric|nonparametric", r"BART", r"Wikipedia|dense (vector )?(index|retriev)|DPR"],
    "Q3": [r"softmax", r"gradient", r"large|grow|magnitude|big"],
    "Q4": [r"RAG-Sequence", r"RAG-Token", r"same (retrieved )?document|single document|one document|same retrieved",
           r"different|each (target |generated )?token|per.token"],
    "Q5": [r"\b8\b|eight", r"\b64\b"],
    "Bonus": [r"BART", r"transformer"],
}


def is_hit(qid: str, answer: str) -> bool:
    return all(re.search(p, answer, flags=re.I) for p in GT_PATTERNS[qid])


EXPECTED_PAPER = {"Q1": "attention_paper.pdf", "Q2": "rag_paper.pdf", "Q3": "attention_paper.pdf",
                  "Q4": "rag_paper.pdf", "Q5": "attention_paper.pdf", "Bonus": "ambos"}
ANSWERS_B1 = {"Q1": answer1, "Q2": answer2, "Q3": answer3, "Q4": answer4, "Q5": answer5, "Bonus": answer_bonus}
QUESTIONS_B1 = {**QUIZ, "Bonus": q_bonus}

b1_rows = []
for qid, question in QUESTIONS_B1.items():
    scored = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=5, score_threshold=0.25)
    time.sleep(0.3)
    n_att = sum(d.metadata["source"] == "attention_paper.pdf" for d, _ in scored)
    n_rag = sum(d.metadata["source"] == "rag_paper.pdf" for d, _ in scored)
    print(f"\n=== {qid} · esperado: {EXPECTED_PAPER[qid]}")
    for d, sc in scored:
        print(f"  {sc:.3f} | {d.metadata['source']:<20} | chunk {d.metadata['chunk_index']:>3} | "
              f"{d.page_content[:70].replace(chr(10), ' ')}")
    exp = EXPECTED_PAPER[qid]
    if exp == "ambos":
        paper_ok = n_att > 0 and n_rag > 0
    else:
        paper_ok = scored[0][0].metadata["source"] == exp
    hit = is_hit(qid, ANSWERS_B1[qid])
    print(f"  → chunks: attention={n_att}, rag={n_rag} | top-1 del paper esperado: {paper_ok} | "
          f"respuesta con todos los datos del ground truth: {hit}")
    b1_rows.append((qid, n_att, n_rag, paper_ok, hit))

print("\nResumen B1 (criterio estricto):", sum(r[4] for r in b1_rows), "de", len(b1_rows))


PLACEHOLDER_B1

---
## B2 · Experimentos con `chunk_size`, `k` y `score_threshold`

Cambio un parámetro a la vez. Primero las funciones auxiliares de la guía (con dos cambios mínimos: indexado por lotes y reintentos), luego los 3 experimentos.

**Criterio de acierto (estricto y constante en las 9 configuraciones):** una pregunta cuenta como acierto si la respuesta contiene **todos** los patrones de `GT_PATTERNS` (la tabla de B1). Si el modelo dice que no tiene evidencia, o responde sin el dato exacto, no cuenta. Es un criterio de palabras clave: no mide estilo, solo si el dato del *ground truth* llegó a la respuesta.

In [ ]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

CORPUS = [
    (attention_text, "Attention Is All You Need", "attention_paper.pdf"),
    (rag_text, "Retrieval-Augmented Generation", "rag_paper.pdf"),
]


def build_store(chunk_size: int, chunk_overlap: int):
    """Chunking + embeddings + Qdrant en memoria. Devuelve (store, n_chunks)."""
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=chunk_size, chunk_overlap=chunk_overlap,
        separators=["\n\n", "\n", ".", " "],
    )
    docs = []
    for text, title, source in CORPUS:
        for i, chunk in enumerate(splitter.split_text(text)):
            if len(chunk.strip()) >= 30:
                docs.append(Document(page_content=chunk,
                                     metadata={"title": title, "source": source, "chunk_index": i}))
    name = f"exp_{chunk_size}_{chunk_overlap}"
    client = QdrantClient(":memory:")
    client.create_collection(collection_name=name,
                             vectors_config=VectorParams(size=EMBEDDING_DIM, distance=Distance.COSINE))
    store = QdrantVectorStore(client=client, collection_name=name, embedding=embeddings)
    add_in_batches(store, docs)          # cambio mío: por lotes y con reintentos
    return store, len(docs)


def build_chain(store, k: int = 5, threshold: float = 0.25):
    """Reutiliza un índice ya construido y cambia solo la recuperación."""
    retr = store.as_retriever(search_type="similarity_score_threshold",
                              search_kwargs={"k": k, "score_threshold": threshold})
    chain = ({"context": retr | format_docs, "question": RunnablePassthrough()}
             | prompt | llm | StrOutputParser())
    return chain, retr


RESULTS = {}      # etiqueta -> dict con chunks del índice, recuperados promedio, aciertos y respuestas


def run_config(label: str, store, n_index: int, k: int, threshold: float):
    chain_, retr_ = build_chain(store, k=k, threshold=threshold)
    hits, retrieved, answers = [], [], {}
    for qid, question in QUIZ.items():
        retrieved.append(len(retrieve(retr_, question)))
        ans = ask(chain_, question)
        answers[qid] = ans
        hits.append(is_hit(qid, ans))
    RESULTS[label] = {"n_index": n_index, "k": k, "th": threshold, "retrieved": retrieved,
                      "avg_ret": sum(retrieved) / len(retrieved), "hits": hits,
                      "n_hits": sum(hits), "answers": answers}
    r = RESULTS[label]
    print(f"{label:<28} índice={n_index:>3} chunks | recuperados por pregunta={retrieved} "
          f"(prom {r['avg_ret']:.1f}) | aciertos={r['n_hits']}/5 {['✔' if h else '✘' for h in hits]}")
    return r


### Experimento 1 · `chunk_size` = 400, 900, 1500 (overlap ≈ 13 %, `k=5`, `score_threshold=0.25`)

Son 3 construcciones del índice (3 pasadas de embeddings sobre el corpus).

In [ ]:
STORES = {}
for cs, ov in [(400, 52), (900, 120), (1500, 200)]:
    t0 = time.time()
    STORES[cs] = build_store(cs, ov)
    print(f"[chunk_size={cs}, overlap={ov}] {STORES[cs][1]} chunks indexados en {time.time() - t0:.0f}s")
    run_config(f"E1 chunk={cs}", STORES[cs][0], STORES[cs][1], k=5, threshold=0.25)


### Experimento 2 · `k` = 3, 5, 8 (reutilizo el índice de 900, `score_threshold=0.25`)

In [ ]:
store_900, n900 = STORES[900]
for k in (3, 5, 8):
    run_config(f"E2 k={k}", store_900, n900, k=k, threshold=0.25)


### Experimento 3 · `score_threshold` = 0.15, 0.25, 0.40 (índice de 900, `k=5`)

In [ ]:
for th in (0.15, 0.25, 0.40):
    run_config(f"E3 threshold={th:.2f}", store_900, n900, k=5, threshold=th)


In [ ]:
print(f"{'Configuración':<26}{'Índice':>7}{'Recup. prom.':>14}{'Aciertos':>10}   por pregunta Q1..Q5")
for label, r in RESULTS.items():
    print(f"{label:<26}{r['n_index']:>7}{r['avg_ret']:>14.1f}{str(r['n_hits']) + '/5':>10}   "
          + " ".join('✔' if h else '✘' for h in r['hits']))

print("\nRespuestas completas por configuración (para revisar a mano los falsos positivos/negativos):")
for label, r in RESULTS.items():
    print(f"\n##### {label}")
    for qid, a in r["answers"].items():
        print(f"[{qid}] {'✔' if is_hit(qid, a) else '✘'} {a.strip().replace(chr(10), ' ')[:420]}")


### Diagnóstico extra: ¿en qué escala está el `score_threshold`?

En los tres umbrales de arriba noté que el número de chunks recuperados no cambia. Reviso los scores crudos para entender por qué: la guía no lo menciona, pero `langchain-qdrant` convierte el coseno `c` a un *score de relevancia* `(c + 1) / 2` antes de comparar con el umbral.

In [ ]:
def relevance_scores(store, question, k=8):
    out = retry_call(store.similarity_search_with_relevance_scores, question, k=k)
    time.sleep(0.3)
    return [sc for _, sc in out]


print("Score de relevancia (coseno+1)/2 de los 8 mejores chunks por pregunta (índice de 900):")
TOP = {}
for qid, question in QUIZ.items():
    TOP[qid] = relevance_scores(store_900, question)
    print(f"  {qid}: " + " ".join(f"{s:.3f}" for s in TOP[qid]))

all_scores = [s for v in TOP.values() for s in v]
print(f"\nMínimo entre los 40 scores: {min(all_scores):.3f} (coseno {2 * min(all_scores) - 1:.3f})")
print("Umbrales del experimento 3 en coseno equivalente: "
      + ", ".join(f"{t:.2f} → {2 * t - 1:.2f}" for t in (0.15, 0.25, 0.40)))

print("\nChunks que sobreviven (k=8) a umbrales que sí actúan sobre esta escala:")
for th in (0.70, 0.75, 0.80, 0.85):
    counts = [sum(s >= th for s in v) for v in TOP.values()]
    print(f"  umbral {th:.2f}: {counts} (prom {sum(counts) / 5:.1f})")


In [ ]:
# Una configuración extra (fuera de las 9 pedidas) con un umbral que sí filtra en esta escala.
run_config("EXTRA threshold=0.80", store_900, n900, k=5, threshold=0.80)


PLACEHOLDER_B2

---
## B3 · Preguntas trampa: ¿el RAG admite que no sabe?

Tres preguntas plausibles cuya respuesta **no está** en los dos papers (modelos y benchmarks posteriores a 2020 o ausentes). Primero compruebo con una búsqueda de texto que los términos clave de verdad no aparecen en el corpus; luego miro qué fragmentos recupera el retriever (si recupera algo con score alto, el modelo tiene "material" para inventar).

In [ ]:
TRAPS = {
    "T1": "What BLEU score did GPT-4 achieve on WMT 2014 English-to-German?",
    "T2": "What exact-match score did RAG obtain on the HotpotQA multi-hop question answering benchmark?",
    "T3": "How many GPUs and how many training days did Llama 2 70B need, and how does that compare with the base Transformer?",
}
full_corpus = (attention_text + "\n" + rag_text).lower()
for term in ("gpt-4", "hotpotqa", "llama", "p100"):
    print(f"¿'{term}' aparece en el corpus? {term in full_corpus}")

PROMPT_V1_SYSTEM = prompt.messages[0].prompt.template
print("\nPROMPT v1 (el del notebook):\n", PROMPT_V1_SYSTEM)

def run_traps(chain_, label):
    out = {}
    for tid, question in TRAPS.items():
        out[tid] = ask(chain_, question)
        print(f"\n[{label}] {tid}: {question}\n→ {out[tid].strip()}")
    return out

print("\n--- Recuperación para cada trampa (score de relevancia) ---")
for tid, question in TRAPS.items():
    scored = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=5, score_threshold=0.25)
    time.sleep(0.3)
    print(f"{tid}: " + " | ".join(f"{sc:.3f} {d.metadata['source'][:4]}#{d.metadata['chunk_index']}" for d, sc in scored))


In [ ]:
# Respuestas con el prompt v1 (el original)
traps_v1 = run_traps(rag_chain, "prompt v1")


PLACEHOLDER_B3a

In [ ]:
# Prompt v2: obliga a comprobar que el dato exacto (modelo, benchmark, métrica) esté en los fragmentos.
PROMPT_V2_SYSTEM = (
    "Answer using ONLY the supplied academic-paper excerpts. "
    "First check whether the excerpts explicitly contain the exact fact the question asks for "
    "(the same model, benchmark and metric). "
    "If they do not, say clearly that the supplied papers do not contain that information. "
    "You may add one short sentence about what related information the excerpts do contain, "
    "but never use outside knowledge, never guess numbers, and never transfer a result from one model "
    "or benchmark to a different one. "
    "Answer in the same language as the question and be concise. "
    "Mention the paper title supporting the answer when there is evidence."
)
prompt_v2 = ChatPromptTemplate.from_messages([
    ("system", PROMPT_V2_SYSTEM),
    ("human", "CONTEXT:\n{context}\n\nQUESTION:\n{question}"),
])
rag_chain_v2 = ({"context": retriever | format_docs, "question": RunnablePassthrough()}
                | prompt_v2 | llm | StrOutputParser())

import difflib
print("DIFF del prompt de sistema (v1 → v2):")
for line in difflib.unified_diff(PROMPT_V1_SYSTEM.split(". "), PROMPT_V2_SYSTEM.split(". "),
                                 "v1", "v2", lineterm="", n=0):
    print(line)

traps_v2 = run_traps(rag_chain_v2, "prompt v2")

print("\n--- Regresión: ¿v2 sigue respondiendo las preguntas legítimas? ---")
for qid in ("Q1", "Q3", "Q5"):
    a = ask(rag_chain_v2, QUIZ[qid])
    print(f"[v2] {qid} {'✔' if is_hit(qid, a) else '✘'} {a.strip().replace(chr(10), ' ')[:300]}")


PLACEHOLDER_B3b

---
## B4 · Preguntas en español sobre papers en inglés

Traduzco Q1, Q3 y Q4. Comparo (a) los chunks recuperados y sus scores, (b) la calidad de la respuesta con los mismos criterios, y (c) hago un mini experimento sobre los embeddings para probar mi hipótesis.

In [ ]:
import numpy as np

QUIZ_ES = {
    "Q1": "¿Qué puntaje BLEU obtuvo el Transformer (modelo grande) en la traducción inglés-alemán de WMT 2014?",
    "Q3": "¿Cuál es el propósito del factor de escala sqrt(dk) en la atención de producto punto escalado?",
    "Q4": "¿Cuáles son las dos variantes del modelo RAG propuestas en el artículo y en qué se diferencian?",
}
# Mismos datos clave que en inglés, pero en español
GT_PATTERNS_ES = {
    "Q1": [r"28[\.,]4"],
    "Q3": [r"softmax", r"gradiente|gradient", r"grande|crec|magnitud|alto|elevad"],
    "Q4": [r"RAG-Sequence", r"RAG-Token", r"mismo documento|un solo documento|único documento|mismo texto recuperado|misma",
           r"diferente|distinto|cada token|por token"],
}
def is_hit_es(qid, answer):
    return all(re.search(p, answer, flags=re.I) for p in GT_PATTERNS_ES[qid])

def top_chunks(question, k=5):
    out = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=k, score_threshold=0.25)
    time.sleep(0.3)
    return [((d.metadata["source"], d.metadata["chunk_index"]), sc) for d, sc in out]

b4 = {}
for qid, q_es in QUIZ_ES.items():
    en, es = top_chunks(QUIZ[qid]), top_chunks(q_es)
    ids_en, ids_es = [c for c, _ in en], [c for c, _ in es]
    inter = set(ids_en) & set(ids_es)
    ans_es = ask(rag_chain, q_es)
    b4[qid] = dict(en=en, es=es, inter=inter, ans_en=ANSWERS_B1[qid], ans_es=ans_es,
                   hit_en=is_hit(qid, ANSWERS_B1[qid]), hit_es=is_hit_es(qid, ans_es))
    print(f"\n=== {qid}\nEN: {QUIZ[qid]}\nES: {q_es}")
    print("  chunks EN:", [(s[:4], i, round(x, 3)) for (s, i), x in en])
    print("  chunks ES:", [(s[:4], i, round(x, 3)) for (s, i), x in es])
    print(f"  chunks en común: {len(inter)} de 5 | top-1 igual: {ids_en[0] == ids_es[0]}")
    print(f"  Respuesta EN ({'✔' if b4[qid]['hit_en'] else '✘'}): {ANSWERS_B1[qid].strip()[:350]}")
    print(f"  Respuesta ES ({'✔' if b4[qid]['hit_es'] else '✘'}): {ans_es.strip()[:350]}")


In [ ]:
# Mini experimento: ¿qué tan cerca quedan en el espacio de embeddings una pregunta y su traducción?
def cos(a, b):
    a, b = np.array(a), np.array(b)
    return float(a @ b / (np.linalg.norm(a) * np.linalg.norm(b)))

def emb_q(text):
    v = retry_call(embeddings.embed_query, text)
    time.sleep(0.3)
    return v

E = {qid: (emb_q(QUIZ[qid]), emb_q(QUIZ_ES[qid])) for qid in QUIZ_ES}
print("Coseno entre la pregunta EN y su traducción ES:")
for qid, (e_en, e_es) in E.items():
    print(f"  {qid}: {cos(e_en, e_es):.3f}")

# Controles: preguntas EN de OTRO tema, y cada ES contra las EN de las otras preguntas
print("\nControl: coseno entre preguntas EN distintas, y ES contra EN de otro tema:")
ids = list(E)
for i in ids:
    for j in ids:
        if i < j:
            print(f"  EN {i} vs EN {j}: {cos(E[i][0], E[j][0]):.3f} | ES {i} vs EN {j}: {cos(E[i][1], E[j][0]):.3f}")

# ¿Qué tan cerca queda cada versión del chunk que contiene el dato? (Q1: tabla con 28.4)
gold = {"Q1": "28.4", "Q4": "RAG-Sequence"}
all_chunks = vectorstore.similarity_search("x", k=1000)  # todo el índice (150 chunks)
for qid, key in gold.items():
    cand = [d for d in all_chunks if key in d.page_content]
    scored = []
    for d in cand[:6]:
        v = emb_q(d.page_content)
        scored.append((cos(E[qid][0], v), cos(E[qid][1], v), d.metadata["source"][:4], d.metadata["chunk_index"]))
    best = max(scored, key=lambda r: max(r[0], r[1]))
    print(f"\n{qid}: chunk con '{key}' mejor puntuado -> coseno EN={best[0]:.3f}, ES={best[1]:.3f} ({best[2]}#{best[3]})")


PLACEHOLDER_B4

---
## B5 · Mi propio quiz con *ground truth*

Leí las secciones de los PDFs y escribí tres preguntas nuevas: una del paper de Attention, una del paper de RAG y una que obliga a combinar los dos. Siguen el formato del notebook (código con la pregunta, bloque `<details>` con el *ground truth* y la sección).

In [ ]:
# ── Pregunta B5-1 (Attention) ─────────────────────────────────
b5q1 = ("How many identical layers are in the encoder and in the decoder stacks of the base Transformer, "
        "and what is the inner-layer dimensionality of the position-wise feed-forward network?")

b5a1 = ask(rag_chain, b5q1)
print(f"Q: {b5q1}")
print(f"A: {b5a1}")


<details>
<summary>✅ Ground Truth — B5-1</summary>

Tanto el encoder como el decoder están formados por una pila de **N = 6** capas idénticas.
La red feed-forward de cada capa tiene dimensionalidad de entrada y salida d_model = 512 y una capa interna de **d_ff = 2048**.
*(Fuente: Sections 3.1 y 3.3 — Attention Is All You Need)*
</details>

In [ ]:
# ── Pregunta B5-2 (RAG) ───────────────────────────────────────
b5q2 = ("Which Wikipedia dump does RAG use as its non-parametric knowledge source, how are the articles split, "
        "and how many documents does the index contain?")

b5a2 = ask(rag_chain, b5q2)
print(f"Q: {b5q2}")
print(f"A: {b5a2}")


<details>
<summary>✅ Ground Truth — B5-2</summary>

RAG usa el volcado de Wikipedia de **diciembre de 2018**. Cada artículo se divide en fragmentos disjuntos de **100 palabras**,
para un total de **21 millones de documentos**. Con el codificador de documentos calculan un embedding por documento y construyen
un índice MIPS con FAISS y una aproximación Hierarchical Navigable Small World.
*(Fuente: Section 3 "Experiments", párrafo inicial — RAG paper)*
</details>

In [ ]:
# ── Pregunta B5-3 (combina los dos papers) ───────────────────
b5q3 = ("Both papers train their models with the Adam optimizer. Which beta values and learning-rate warmup "
        "does the Transformer paper use, and which component of RAG is kept fixed (not fine-tuned) during training?")

b5a3 = ask(rag_chain, b5q3)
print(f"Q: {b5q3}")
print(f"A: {b5a3}")


<details>
<summary>✅ Ground Truth — B5-3 (combina los dos papers)</summary>

**Attention Is All You Need (Sección 5.3):** Adam con β1 = 0.9, β2 = 0.98 y ε = 10^-9; la tasa de aprendizaje crece linealmente durante
**warmup_steps = 4000** pasos y luego decrece proporcionalmente a la raíz cuadrada inversa del paso.
**RAG:** también entrena con Adam, pero mantiene **fijos el codificador de documentos (BERT_d) y, con él, el índice de documentos**;
solo ajusta el codificador de consultas (BERT_q) y el generador BART, porque actualizar el índice sería costoso.
*(Fuentes: Section 5.3 del Transformer y Section 3 — Experiments del RAG paper)*
</details>

In [ ]:
B5 = {
    "B5-1": (b5q1, b5a1, [r"\b6\b|six", r"2048"], "attention_paper.pdf"),
    "B5-2": (b5q2, b5a2, [r"December 2018|2018", r"100.word", r"21 ?M|21 million"], "rag_paper.pdf"),
    "B5-3": (b5q3, b5a3, [r"0\.98", r"4000|4,000", r"document encoder|BERT.?d|index"], "ambos"),
}
for bid, (question, ans, pats, exp) in B5.items():
    scored = retry_call(vectorstore.similarity_search_with_relevance_scores, question, k=5, score_threshold=0.25)
    time.sleep(0.3)
    n_att = sum(d.metadata["source"] == "attention_paper.pdf" for d, _ in scored)
    n_rag = sum(d.metadata["source"] == "rag_paper.pdf" for d, _ in scored)
    hit = all(re.search(p, ans, flags=re.I) for p in pats)
    print(f"\n=== {bid} · paper esperado: {exp}")
    for d, sc in scored:
        print(f"  {sc:.3f} | {d.metadata['source']:<20} | chunk {d.metadata['chunk_index']:>3} | "
              f"{d.page_content[:70].replace(chr(10), ' ')}")
    print(f"  chunks: attention={n_att}, rag={n_rag} | datos clave completos en la respuesta: {hit}")


PLACEHOLDER_B5

---
## Reflexión final

1. **¿Qué ocurrió cuando el contexto no contenía información suficiente?** ¿El modelo reconoció la limitación o inventó una respuesta?
2. **¿Qué fragmentos recuperó para la pregunta bonus?** ¿Aparecieron ambos papers?
3. **¿Qué efecto tendría modificar `chunk_size`, `chunk_overlap`, `k` o `score_threshold`?**
4. **¿Por qué conviene separar el modelo de embeddings del modelo generativo?**
5. **¿Qué cambiarías para llevar este prototipo a producción?** Considera persistencia, control de acceso, evaluación, observabilidad y costos.


PLACEHOLDER_R1